# PATHDEFENSE Demo
Demonstration notebook for "Defense Against Shortest Path Attacks"

This material is based upon work supported by the United States Air Force under
Air  Force  Contract  No.  FA8702-15-D-0001  and  the  Combat  Capabilities
Development Command Army Research Laboratory (under Cooperative Agreement Number
W911NF-13-2-0045).  Any  opinions,  findings,  conclusions  or  recommendations
expressed in this material are those of the authors and do not necessarily
reflect the views of the United States Air Force or Army Research Laboratory.

Copyright (C) 2023
Benjamin A. Miller, Zohair Shafi, Wheeler Ruml, Yevgeniy Vorobeychik, Tina Eliassi-Rad, and Scott Alfeld

The software is provided to you on an As-Is basis

Delivered to the U.S. Government with Unlimited Rights, as defined in DFARS Part
252.227-7013 or 7014 (Feb 2014). Notwithstanding any copyright notice, U.S.
Government rights in this work are defined by DFARS 252.227-7013 or DFARS
252.227-7014 as detailed above. Use of this work other than as specifically
authorized by the U.S. Government may violate any copyrights that exist in this
work

In [ ]:
from opt_defense import zero_sum_heuristic as ZeroSum
from opt_defense import increment_edge_heuristic as PATHDEFENSE

import networkx as nx
import numpy as np
import numpy.random as rand
from numpy import linalg as la
import random
from scipy import stats

import gurobipy as gp
from gurobipy import GRB

import matplotlib as mpl
from matplotlib import pyplot as plt

In [ ]:
mpl.rc('xtick', labelsize=14, color="#222222")
mpl.rc('ytick', labelsize=14, color="#222222")
mpl.rc('font', **{'family':'sans-serif','sans-serif':['Arial']})
mpl.rc('font', size=16)
mpl.rc('xtick.major', size=6, width=1)
mpl.rc('xtick.minor', size=3, width=1)
mpl.rc('ytick.major', size=6, width=1)
mpl.rc('ytick.minor', size=3, width=1)
mpl.rc('axes', linewidth=1, edgecolor="#222222", labelcolor="#222222")
mpl.rc('text', usetex=False, color="#222222")

## Create graph
Create a 50-node Watts-Strogatz graph with Poisson weights

In [ ]:
G = nx.watts_strogatz_graph(50, 4, .1)

for e in G.edges:
    G.edges[e]['cost'] = 1
    G.edges[e]['weight'] = 1+rand.poisson(20)

lcc = list(max(nx.connected_components(G), key=len))
G = nx.subgraph(G, lcc)

## Get target paths
Choose two nodes at random and the 5th, 7th, and 9th paths between them as the targets, each with equal probability

In [ ]:
pathDict = {}
st = rand.choice(lcc, size=2, replace=False)
s = st[0]
t = st[1]
ctr = 0
temp_paths = []
for p in nx.shortest_simple_paths(G, s, t, 'weight'):
    ctr += 1
    temp_paths.append(p)
    if ctr >= 9:
        break
        
# try again if there weren't enough paths
while ctr < 9:
    st = rand.choice(lcc, size=2, replace=False)
    s = st[0]
    t = st[1]
    ctr = 0
    temp_paths = []
    for p in nx.shortest_simple_paths(G, s, t, 'weight'):
        ctr += 1
        temp_paths.append(p)
        if ctr >= 9:
            break
# pick the 5th, 7th, and 9th paths as targets
for ii in range(3):
    pathDict[tuple(temp_paths[5+2*ii-1])] = 1/3

## Distributions and variabless
Buget distribution is a truncated Poissson

Source–destination distribution makes probability proportional to degree

marginal penalty for overstating length: 0.1

marginal penalty for understating length: 0.8

external cost of successful attack: 5

In [ ]:
#budget distribution
B = stats.poisson.pmf(np.arange(10), 6)
B = B/np.sum(B)

#source/destination distribution
D = {}
pairNorm = 0
deg = nx.degree(G)
V = list(G.nodes())
for ii in range(len(G)):
    D[V[ii]] = {}
    di = nx.degree(G, V[ii])
    for jj in range(ii+1, len(G)):
        #probability proportional to degree
        D[V[ii]][V[jj]] = deg(ii)*deg(jj) 
        pairNorm += D[V[ii]][V[jj]]
#normalize distribution
for ii in range(len(G)):
    for jj in range(ii+1, len(G)):
            D[V[ii]][V[jj]] /= pairNorm

fminus = .8
fplus = .1
lmda = 5


## Run PATHDEFENSE

In [ ]:
result_PATHDEFENSE = PATHDEFENSE(G, D, B, pathDict, fminus, fplus, lmda, 0, max_iter=100)

## Plot results

In [ ]:
cost_PATHDEFENSE = np.array(result_PATHDEFENSE[1])
plt.plot(cost_PATHDEFENSE)
plt.plot(np.sum(cost_PATHDEFENSE, axis=1))
plt.legend(['Distance', 'Error', 'Attack Success', 'All'])
plt.xlabel('Iteration')
plt.ylabel('Defender Cost')
plt.title('PATHDEFENSE Results')
plt.show()